# 00 — Data validation

This notebook freezes the BTCUSDT one-minute research dataset. It is not a trading hypothesis and it does not read the sealed test as evidence.

The AWS profile is `trading`. The default credential chain can point at an account that does not own `trading-data-centheos`.

The load reads the legacy archive `ohlcv/binance/BTCUSDT/1m.parquet` and the yearly shards under `1m/`. The collector's yearly reader does not open the archive, so this notebook does not use that reader.


In [1]:
from pathlib import Path
import os
import sys

root = Path.cwd().resolve()
while not (root / "research" / "__init__.py").is_file() and root != root.parent:
    root = root.parent
if not (root / "research" / "__init__.py").is_file():
    raise RuntimeError("open this notebook from the trading repository")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["AWS_PROFILE"] = "trading"
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
os.environ["DATA_STORE"] = "s3"
os.environ["S3_BUCKET"] = "trading-data-centheos"


## Checks

The audit counts duplicate timestamps, off-grid timestamps, non-positive prices, inconsistent OHLC, and missing minutes. Missing minutes stay missing. The manifest digest covers the S3 object identities and those exclusion counts.


In [2]:
import pandas as pd

from research.evaluate import freeze_dataset
from research.load import load_research_bars
from research.manifest import load_manifest, save_manifest
from research.protocol import H1_CONFIG

manifest_path = root / "research" / "registry" / "datasets" / "btcusdt-1m.json"
bars, objects = load_research_bars(
    venue=H1_CONFIG["venue"],
    symbol=H1_CONFIG["symbol"],
    timeframe=H1_CONFIG["timeframe"],
    bucket=H1_CONFIG["bucket"],
    profile=H1_CONFIG["profile"],
    region=H1_CONFIG["region"],
)
if bars.empty:
    raise RuntimeError("BTCUSDT one-minute load returned no rows")
kept, manifest, report = freeze_dataset(
    bars,
    objects,
    dataset_id="btcusdt-1m",
    venue=H1_CONFIG["venue"],
    symbol=H1_CONFIG["symbol"],
    timeframe=H1_CONFIG["timeframe"],
    bucket=H1_CONFIG["bucket"],
    profile=H1_CONFIG["profile"],
)
save_manifest(manifest_path, manifest)
loaded = load_manifest(manifest_path)
if loaded.digest != manifest.digest:
    raise RuntimeError("saved dataset manifest did not reload with the same digest")
print(f"rows_in={report.rows_in} rows_out={report.rows_out} digest={manifest.digest}")
report.exclusions


rows_in=3585983 rows_out=3585983 digest=d67871b2eebd796cb37c7cafa89677fbca4b31ad0bdae959a5f0e2efeecc6c88


QualityExclusions(duplicate_timestamps=0, off_grid_timestamps=0, nonpositive_prices=0, inconsistent_ohlc=0, missing_minutes=0)